In [ ]:
"""
TorajiGrade - AI Inference & Metadata Engine

Computer Vision inference engine for:
TorajiGrade: Penilai Konsistensi Tingkat Sangrai
dan Deteksi Biji Mentah Kopi Toraja.

Architecture:
    MobileNetV3-Small

Classes:
    0 -> Dark
    1 -> Green
    2 -> Light
    3 -> Medium

Inference:
    CPU-only
    Synchronous
    PyTorch
"""

In [ ]:
from __future__ import annotations

import io
import json
from pathlib import Path
from typing import Any

import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from torchvision import models, transforms


In [ ]:

class TorajiGradeInference:
    """
    Production-oriented inference engine for TorajiGrade.

    The model is loaded directly onto CPU and inference
    is performed synchronously without gradient computation.
    """

    # ========================================================
    # Class configuration
    # ========================================================

    CLASS_NAMES = [
        "Dark",
        "Green",
        "Light",
        "Medium",
    ]

    # ========================================================
    # Commercial metadata
    # ========================================================

    METADATA = {

        "Green": {
            "description": (
                "Biji kopi masih dalam kondisi mentah "
                "(Green Beans). Belum melalui proses penyangraian."
            ),

            "action": (
                "Saring kembali biji dari kotoran/batu kecil, "
                "pastikan kadar air berkisar 11-12% sebelum "
                "dimasukkan ke dalam hopper mesin roasting."
            ),
        },

        "Light": {
            "description": (
                "Tingkat sangrai Light (Muda). Warna cokelat terang, "
                "body ringan, dengan keasaman (acidity) khas "
                "Arabika Toraja yang sangat menonjol."
            ),

            "action": (
                "Ideal untuk metode seduh manual (Filter/V60) "
                "untuk mengekstraksi rasa buah (fruity) khas Toraja. "
                "Simpan dalam wadah degassing selama 3-5 hari."
            ),
        },

        "Medium": {
            "description": (
                "Tingkat sangrai Medium (Sedang). Keseimbangan "
                "sempurna antara tingkat keasaman (acidity) dan "
                "rasa manis (sweetness), warna cokelat keemasan seragam."
            ),

            "action": (
                "Sangat direkomendasikan untuk konsumsi harian "
                "toko kopi lokal (Espresso base / Kopi Susu kekinian). "
                "Kualitas sangrai berada pada tingkat paling stabil."
            ),
        },

        "Dark": {
            "description": (
                "Tingkat sangrai Dark (Tua). Warna cokelat sangat "
                "gelap hingga kehitaman, permukaan mulai berminyak, "
                "aroma gosong/smoky yang kuat dengan keasaman minimal."
            ),

            "action": (
                "Cocok digunakan untuk racikan kopi tradisional "
                "(Kopi Tubruk Toraja asli) atau blend espresso "
                "yang membutuhkan rasa pahit bold dan body tebal."
            ),
        },
    }

    # ========================================================
    # ImageNet preprocessing
    # ========================================================

    IMAGE_SIZE = (224, 224)

    IMAGENET_MEAN = [
        0.485,
        0.456,
        0.406,
    ]

    IMAGENET_STD = [
        0.229,
        0.224,
        0.225,
    ]

    def __init__(
        self,
        model_path: str | Path,
        config_path: str | Path | None = None,
    ) -> None:
        """
        Initialize the TorajiGrade inference engine.

        Parameters
        ----------
        model_path:
            Path to best_torajigrade_model.pth.

        config_path:
            Optional path to torajigrade_model_config.json.
        """

        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"Model tidak ditemukan: {self.model_path}"
            )

        # ----------------------------------------------------
        # CPU-only inference
        # ----------------------------------------------------

        self.device = torch.device("cpu")

        # ----------------------------------------------------
        # Load optional configuration
        # ----------------------------------------------------

        self.config: dict[str, Any] = {}

        if config_path is not None:
            config_path = Path(config_path)

            if config_path.exists():
                with open(
                    config_path,
                    "r",
                    encoding="utf-8",
                ) as f:
                    self.config = json.load(f)

        # ----------------------------------------------------
        # Build architecture
        # ----------------------------------------------------

        self.model = self._build_model()

        # ----------------------------------------------------
        # Load trained weights
        # ----------------------------------------------------

        self._load_weights()

        # ----------------------------------------------------
        # Validation preprocessing
        # ----------------------------------------------------

        self.transform = transforms.Compose(
            [
                transforms.Resize(self.IMAGE_SIZE),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=self.IMAGENET_MEAN,
                    std=self.IMAGENET_STD,
                ),
            ]
        )

    # ========================================================
    # Model construction
    # ========================================================

    def _build_model(self) -> nn.Module:
        """
        Build the exact MobileNetV3-Small architecture
        used during training.
        """

        model = models.mobilenet_v3_small(
            weights=None
        )

        # Replace final classifier with 4-class output.
        in_features = model.classifier[3].in_features

        model.classifier[3] = nn.Linear(
            in_features,
            len(self.CLASS_NAMES),
        )

        return model

    # ========================================================
    # Load weights
    # ========================================================

    def _load_weights(self) -> None:
        """
        Load trained model weights directly onto CPU.
        """

        checkpoint = torch.load(
            self.model_path,
            map_location=torch.device("cpu"),
        )

        # ----------------------------------------------------
        # Support standard state_dict checkpoints
        # ----------------------------------------------------

        if isinstance(checkpoint, dict):

            # Standard state_dict
            if all(
                isinstance(key, str)
                for key in checkpoint.keys()
            ):
                state_dict = checkpoint

            # Checkpoint containing state_dict
            elif "state_dict" in checkpoint:
                state_dict = checkpoint["state_dict"]

            elif "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]

            else:
                raise RuntimeError(
                    "Format checkpoint tidak dikenali."
                )

        else:
            raise RuntimeError(
                "Checkpoint model bukan dictionary/state_dict."
            )

        self.model.load_state_dict(
            state_dict,
            strict=True,
        )

        self.model.to(self.device)

        self.model.eval()

In [ ]:
    # ========================================================
    # Image preprocessing
    # ========================================================

    def _preprocess(
        self,
        image: Image.Image,
    ) -> torch.Tensor:
        """
        Apply validation/test preprocessing.
        """

        image = image.convert("RGB")

        tensor = self.transform(image)

        # Add batch dimension.
        tensor = tensor.unsqueeze(0)

        return tensor.to(self.device)

In [ ]:
    # ========================================================
    # Prediction
    # ========================================================

    def predict_image(
        self,
        image_bytes: bytes,
    ) -> dict[str, Any]:
        """
        Predict coffee bean class from raw image bytes.

        Parameters
        ----------
        image_bytes:
            Raw bytes of an image file.

        Returns
        -------
        dict
            Prediction result including:
            - predicted class
            - class index
            - confidence
            - all class probabilities
            - commercial description
            - recommended action
        """

        if not image_bytes:
            raise ValueError(
                "image_bytes kosong."
            )

        # ----------------------------------------------------
        # Open image from bytes
        # ----------------------------------------------------

        try:
            image = Image.open(
                io.BytesIO(image_bytes)
            )

            image.load()

        except (
            UnidentifiedImageError,
            OSError,
        ) as exc:

            raise ValueError(
                "File bukan gambar yang valid "
                "atau gambar tidak dapat dibaca."
            ) from exc

        # ----------------------------------------------------
        # Preprocess
        # ----------------------------------------------------

        input_tensor = self._preprocess(image)

        # ----------------------------------------------------
        # Inference
        # ----------------------------------------------------

        with torch.no_grad():

            logits = self.model(
                input_tensor
            )

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        confidence_tensor, predicted_tensor = torch.max(
            probabilities,
            dim=1,
        )

        predicted_index = int(
            predicted_tensor.item()
        )

        confidence = float(
            confidence_tensor.item()
        )

        predicted_class = self.CLASS_NAMES[
            predicted_index
        ]

        # ----------------------------------------------------
        # All class probabilities
        # ----------------------------------------------------

        probability_values = probabilities[
            0
        ].cpu().tolist()

        probabilities_dict = {
            class_name: round(
                probability * 100,
                4,
            )
            for class_name, probability
            in zip(
                self.CLASS_NAMES,
                probability_values,
            )
        }

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        metadata = self.METADATA[
            predicted_class
        ]

        # ----------------------------------------------------
        # Final result
        # ----------------------------------------------------

        return {
            "success": True,

            "prediction": {
                "class": predicted_class,
                "class_index": predicted_index,
                "confidence": round(
                    confidence * 100,
                    2,
                ),
            },

            "probabilities": probabilities_dict,

            "metadata": {
                "description": metadata[
                    "description"
                ],

                "action": metadata[
                    "action"
                ],
            },

            "model": {
                "architecture": "MobileNetV3-Small",
                "input_size": [
                    224,
                    224,
                ],
                "device": "cpu",
            },
        }

In [ ]:
# ============================================================
# Standalone Colab test
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TorajiGrade - Inference Engine")
    print("=" * 60)

    # --------------------------------------------------------
    # Default Colab / Google Drive paths
    # --------------------------------------------------------

    DATASET_ROOT = Path(
        "/content/drive/MyDrive/Dataset/dataset-AIC"
    )

    MODEL_PATH = (
        DATASET_ROOT
        / "best_torajigrade_model.pth"
    )

    CONFIG_PATH = (
        DATASET_ROOT
        / "torajigrade_model_config.json"
    )

    # --------------------------------------------------------
    # Check model
    # --------------------------------------------------------

    if not MODEL_PATH.exists():

        raise FileNotFoundError(
            f"""
Model tidak ditemukan:

{MODEL_PATH}

Pastikan Google Drive sudah di-mount
dan file best_torajigrade_model.pth
tersedia.
"""
        )

    # --------------------------------------------------------
    # Initialize engine
    # --------------------------------------------------------

    engine = TorajiGradeInference(
        model_path=MODEL_PATH,
        config_path=CONFIG_PATH,
    )

    print(
        f"Model : {MODEL_PATH}"
    )

    print(
        "Device: CPU"
    )

    print(
        "Status: READY"
    )

    print("=" * 60)